# Pharma Weekly Enhancement — Final Category-Specific Combination

Notebook final untuk menjalankan kombinasi terbaik per kategori dari hasil eksperimen
baseline + enhancement. Struktur runtime mengikuti notebook enhancement: Colab-ready,
install dependencies, GEO dual-mode, output fallback ke Drive/MyDrive, dan checkpoint/resume.

Eksperimen ini hanya menjalankan konfigurasi final terpilih per kategori (8 langkah utama),
bukan semua 4 optimizer x 2 skema x 8 kategori lagi.

# Runtime Configuration

Set `USE_GOOGLE_COLAB` untuk memilih runtime:

- `False` — lokal / JupyterHub: load `salesweekly.csv` dari `data/raw/pharma-sales`
  di bawah project root.
- `True` — Google Colab: mount Drive dan load dari path Shared Drive.

`QUICK_MODE` mengecilkan jumlah iterasi optimizer untuk uji cepat. Set `False`
untuk run final (iterasi penuh seperti penelitian).

In [1]:
# ============================================================
# Runtime toggle - set sebelum menjalankan cell setup di bawah
# ============================================================
USE_GOOGLE_COLAB = True   # True = Colab + Shared Drive; False = lokal / JupyterHub
QUICK_MODE = False          # True = iterasi kecil (uji cepat); False = iterasi penuh

# Path Shared Drive saat USE_GOOGLE_COLAB=True (folder berisi salesweekly.csv)
COLAB_WEEKLY_DIR = (
    '/content/drive/Shareddrives/Riset S3/Forecasting/'
    'dataset-drug-demand-prediction/pharmasales'
)

print(f'USE_GOOGLE_COLAB = {USE_GOOGLE_COLAB}')
print(f'QUICK_MODE       = {QUICK_MODE}')
if USE_GOOGLE_COLAB:
    print(f'Colab weekly dir : {COLAB_WEEKLY_DIR}')
else:
    print('Dataset akan diresolusi dari project root: data/raw/pharma-sales')

USE_GOOGLE_COLAB = True
QUICK_MODE       = False
Colab weekly dir : /content/drive/Shareddrives/Riset S3/Forecasting/dataset-drug-demand-prediction/pharmasales


## Install Dependencies

Di Google Colab, `optuna` dan `pyswarms` belum terpasang secara default. Cell
berikut menginstalnya saat berjalan di Colab (dilewati di lokal / JupyterHub yang
sudah punya dependency).

In [2]:
# Install dependency yang tidak tersedia default di Colab
import importlib, subprocess, sys

_pkgs = {'optuna': 'optuna', 'pyswarms': 'pyswarms', 'xgboost': 'xgboost',
         'statsmodels': 'statsmodels'}
_missing = [pip_name for mod, pip_name in _pkgs.items()
            if importlib.util.find_spec(mod) is None]
if _missing:
    print('Installing:', _missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=True)
    print('Selesai install.')
else:
    print('Semua dependency sudah tersedia.')

Installing: ['optuna', 'pyswarms']
Selesai install.


In [3]:
import os
import sys
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import acf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
import xgboost as xgb
import optuna
from pyswarms.single.global_best import GlobalBestPSO

optuna.logging.set_verbosity(optuna.logging.ERROR)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
sns.set_theme(style="whitegrid")


def resolve_project_root() -> Path:
    """Resolusi repo root saat dijalankan dari notebooks/ atau sejenisnya."""
    root = Path(os.getcwd()).resolve()
    if root.name.lower() in {'explore', 'notebooks', 'notebooks-to-transfers'}:
        root = root.parent
        if root.name.lower() == 'notebooks':
            root = root.parent
    return root


def configure_colab_runtime(use_google_colab: bool, colab_weekly_dir: str):
    """Mount Drive dan kembalikan dir dataset saat Colab; else None."""
    if not use_google_colab:
        return None
    try:
        from google.colab import drive
    except ImportError as exc:
        raise ImportError(
            'USE_GOOGLE_COLAB=True tetapi google.colab tidak tersedia. '
            'Jalankan di Google Colab, atau set USE_GOOGLE_COLAB=False.'
        ) from exc
    drive.mount('/content/drive')
    data_dir = Path(colab_weekly_dir)
    print('[INFO] Google Colab mode aktif.')
    print(f'[INFO] Drive mounted; weekly dir -> {data_dir}')
    return data_dir


if 'USE_GOOGLE_COLAB' not in globals():
    raise NameError('USE_GOOGLE_COLAB belum didefinisikan. Jalankan cell Runtime Configuration dulu.')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

COLAB_DATA_DIR = configure_colab_runtime(USE_GOOGLE_COLAB, COLAB_WEEKLY_DIR)

if USE_GOOGLE_COLAB:
    DATA_DIR = COLAB_DATA_DIR if COLAB_DATA_DIR is not None else Path(COLAB_WEEKLY_DIR)
else:
    DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'pharma-sales'

WEEKLY_PATH = DATA_DIR / 'salesweekly.csv'
if not WEEKLY_PATH.exists():
    raise FileNotFoundError(f'salesweekly.csv tidak ditemukan: {WEEKLY_PATH}')

if USE_GOOGLE_COLAB:
    _output_candidates = [
        DATA_DIR / 'weekly_enhancement_outputs',
        Path('/content/drive/MyDrive/pharma_weekly_enhancement_outputs'),
        Path('/content/pharma_weekly_enhancement_outputs'),
    ]
else:
    _output_candidates = [PROJECT_ROOT / 'outputs' / 'enhancements']

OUTPUT_DIR = None
_last_output_error = None
for _candidate in _output_candidates:
    try:
        _candidate.mkdir(parents=True, exist_ok=True)
        _probe = _candidate / '.write_probe'
        _probe.write_text('ok', encoding='utf-8')
        _probe.unlink(missing_ok=True)
        OUTPUT_DIR = _candidate
        break
    except OSError as exc:
        _last_output_error = exc

if OUTPUT_DIR is None:
    raise OSError(f'Tidak ada output directory yang writable. Last error: {_last_output_error}')
if USE_GOOGLE_COLAB and OUTPUT_DIR != _output_candidates[0]:
    print(f'[WARN] Shared Drive output read-only. Fallback output dir -> {OUTPUT_DIR}')

print(f'Runtime mode  : {"Google Colab" if USE_GOOGLE_COLAB else "Lokal / JupyterHub"}')
print(f'Project root  : {PROJECT_ROOT}')
print(f'Data path     : {WEEKLY_PATH}')
print(f'Output dir    : {OUTPUT_DIR}')

# Deteksi GPU XGBoost (opsional, aman jika tidak ada)
try:
    _probe = xgb.XGBRegressor(tree_method='hist', device='cuda', n_estimators=1)
    _probe.fit(np.zeros((4, 2)), np.zeros(4))
    XGB_DEVICE = 'cuda'
except Exception:
    XGB_DEVICE = 'cpu'
print(f'XGBoost device: {XGB_DEVICE}')

Mounted at /content/drive
[INFO] Google Colab mode aktif.
[INFO] Drive mounted; weekly dir -> /content/drive/Shareddrives/Riset S3/Forecasting/dataset-drug-demand-prediction/pharmasales
[WARN] Shared Drive output read-only. Fallback output dir -> /content/drive/MyDrive/pharma_weekly_enhancement_outputs
Runtime mode  : Google Colab
Project root  : /content
Data path     : /content/drive/Shareddrives/Riset S3/Forecasting/dataset-drug-demand-prediction/pharmasales/salesweekly.csv
Output dir    : /content/drive/MyDrive/pharma_weekly_enhancement_outputs
XGBoost device: cuda


## Load class GEO (dual-mode)

Lokal: import dari `src/utils/geo.py` (single source of truth di repo).
Colab / standalone: fallback ke definisi inline yang identik dengan file repo,
sehingga notebook tetap jalan tanpa perlu meng-upload file terpisah.

In [4]:
try:
    from src.utils.geo import GEO
    print('[INFO] GEO di-load dari src/utils/geo.py (repo lokal).')
except ImportError:
    # ---- Fallback inline (identik dengan src/utils/geo.py) ----
    class GEO:
        def __init__(self, obj_func, dim, bounds, n_agents=10, max_iter=50):
            self.obj_func = obj_func
            self.dim = dim
            self.bounds = bounds
            self.n_agents = n_agents
            self.max_iter = max_iter

        def optimize(self):
            X = np.random.rand(self.n_agents, self.dim)
            for i in range(self.dim):
                lb, ub = self.bounds[i]
                X[:, i] = lb + (ub - lb) * X[:, i]

            fitness = np.array([self.obj_func(x) for x in X])
            gbest = X[np.argmin(fitness)].copy()
            gbest_fit = np.min(fitness)

            for t in range(self.max_iter):
                alpha = 2 * (1 - t / self.max_iter)
                for i in range(self.n_agents):
                    r1, r2 = np.random.rand(), np.random.rand()
                    A = 2 * alpha * r1 - alpha
                    C = 2 * r2
                    D = abs(C * gbest - X[i])
                    newX = gbest - A * D
                    for d in range(self.dim):
                        lb, ub = self.bounds[d]
                        newX[d] = np.clip(newX[d], lb, ub)
                    new_fit = self.obj_func(newX)
                    if new_fit < fitness[i]:
                        X[i] = newX
                        fitness[i] = new_fit
                if np.min(fitness) < gbest_fit:
                    gbest = X[np.argmin(fitness)].copy()
                    gbest_fit = np.min(fitness)
            return gbest, gbest_fit
    print('[INFO] GEO di-load dari fallback inline (Colab / standalone).')

[INFO] GEO di-load dari fallback inline (Colab / standalone).


In [5]:
# Konfigurasi iterasi optimizer berdasarkan QUICK_MODE
if QUICK_MODE:
    OPT_CFG = {'optuna_trials': 15, 'pso_iters': 10, 'pso_particles': 10,
               'geo_iters': 10, 'geo_agents': 8}
else:
    OPT_CFG = {'optuna_trials': 50, 'pso_iters': 30, 'pso_particles': 20,
               'geo_iters': 30, 'geo_agents': 15}
print('OPT_CFG =', OPT_CFG)

OPT_CFG = {'optuna_trials': 50, 'pso_iters': 30, 'pso_particles': 20, 'geo_iters': 30, 'geo_agents': 15}


# Load Data

In [6]:
data = pd.read_csv(WEEKLY_PATH)
categories = ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']
data['datum'] = pd.to_datetime(data['datum'])
print('Shape:', data.shape)
data.head()

Shape: (302, 9)


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,2014-01-05,14.00,11.67,21.3,185.95,41.0,0.0,32.0,7.0
1,2014-01-12,29.33,12.68,37.9,190.70,88.0,5.0,21.0,7.2
2,2014-01-19,30.67,26.34,45.9,218.40,80.0,8.0,29.0,12.0
3,2014-01-26,34.00,32.37,31.5,179.60,80.0,8.0,23.0,10.0
4,2014-02-02,31.02,23.35,20.7,159.88,84.0,12.0,29.0,12.0


# Feature Engineering (ACF-based)

Pemilihan lag memakai **ACF** (sesuai konvensi eksperimen daily). Untuk tiap
kategori, hitung ACF sampai `nlags=26`, ambil lag yang **signifikan** (di atas
pita kepercayaan 95% ~ `1.96/sqrt(N)`). Baseline (Section 1) memakai lag ACF +
rolling mean; Section 2 menambah fitur musiman.

In [7]:
ACF_NLAGS = 26

def select_acf_lags(series, nlags=ACF_NLAGS):
    """Kembalikan daftar lag signifikan (>95% CI) + lag maksimum untuk rolling."""
    vals = acf(series.dropna(), nlags=nlags, fft=False)
    n = len(series.dropna())
    conf = 1.96 / np.sqrt(n)
    sig = [lag for lag in range(1, nlags + 1) if abs(vals[lag]) > conf]
    if not sig:                       # fallback: lag dgn |ACF| tertinggi
        sig = [int(np.argmax(np.abs(vals[1:])) + 1)]
    return sig, max(sig)


def build_features(df, category, seasonal=False):
    """Bangun DataFrame fitur untuk satu kategori.

    seasonal=False -> baseline: lag ACF signifikan + 1 rolling mean.
    seasonal=True  -> tambah Fourier(52), kalender, rolling std/min/max, diff.
    """
    dfg = df[['datum', category]].rename(columns={'datum': 'ds', category: 'y'}).copy()
    dfg['ds'] = pd.to_datetime(dfg['ds'])

    sig_lags, max_lag = select_acf_lags(dfg['y'])
    for lag in sig_lags:
        dfg[f'lag_{lag}'] = dfg['y'].shift(lag)
    dfg[f'rolling_mean_{max_lag}'] = dfg['y'].shift(1).rolling(window=max_lag).mean()

    if seasonal:
        # Fourier terms untuk siklus tahunan (~52 minggu)
        woy = dfg['ds'].dt.isocalendar().week.astype(int)
        for k in (1, 2, 3):
            dfg[f'fourier_sin_{k}'] = np.sin(2 * np.pi * k * woy / 52.0)
            dfg[f'fourier_cos_{k}'] = np.cos(2 * np.pi * k * woy / 52.0)
        dfg['week_of_year'] = woy
        dfg['month'] = dfg['ds'].dt.month
        dfg['quarter'] = dfg['ds'].dt.quarter
        dfg[f'rolling_std_{max_lag}'] = dfg['y'].shift(1).rolling(max_lag).std()
        dfg[f'rolling_min_{max_lag}'] = dfg['y'].shift(1).rolling(max_lag).min()
        dfg[f'rolling_max_{max_lag}'] = dfg['y'].shift(1).rolling(max_lag).max()
        dfg['diff_1'] = dfg['y'].diff().shift(1)

    dfg = dfg.dropna().reset_index(drop=True)
    return dfg


def make_splits(df, seasonal=False):
    """Bangun data per kategori: train/val/test (70/10/20 kronologis)."""
    out = []
    for category in categories:
        dfg = build_features(df, category, seasonal=seasonal)
        n = len(dfg)
        train_size = int(n * 0.7)
        val_size = int(n * 0.1)
        dfgtrain = dfg.iloc[:train_size]
        dfgval = dfg.iloc[train_size:train_size + val_size]
        dfgtest = dfg.iloc[train_size + val_size:]
        feat_cols = [c for c in dfg.columns if c not in ('ds', 'y')]
        out.append({
            'category': category,
            'dfgTest': dfgtest,
            'n_features': len(feat_cols),
            'data': {
                'X_train': dfgtrain[feat_cols].to_numpy(),
                'y_train': dfgtrain['y'].to_numpy(),
                'X_val': dfgval[feat_cols].to_numpy(),
                'y_val': dfgval['y'].to_numpy(),
                'X_test': dfgtest[feat_cols].to_numpy(),
                'y_test': dfgtest['y'].to_numpy(),
            }
        })
    return out

## Mesin eksperimen

`run_experiment(cfg, feature_sets, label)` menjalankan **4 optimizer × 2 skema
hybrid** untuk 8 kategori. Semua optimizer **meminimalkan RMSE validation**;
model terbaik lalu di-fit ulang di train dan dievaluasi di **test** (MSE & RMSE).

`cfg` (dict) mengontrol tiap enhancement:
- `xgb_space`: `'base'` (3 HP) atau `'expanded'` (8 HP).
- `early_stopping`: bool.
- `target_transform`: `None` atau `'log1p'` (model di ruang tertransformasi).
- `xgb_objective`: `'reg:squarederror'` | `'count:poisson'` | `'reg:tweedie'` | `'reg:pseudohubererror'`.
- `winsorize`: `None` atau `(low_pct, high_pct)` clip target train.

Catatan: untuk `count:poisson`/`reg:tweedie` pada skema **residual**, target
residual yang negatif di-clip ke 0 (objective ini butuh target non-negatif) —
ini simplifikasi yang didokumentasikan.

In [8]:
PARAM_ORDER_BASE = ['n_estimators', 'max_depth', 'learning_rate']
PARAM_EXTRA = ['reg_alpha', 'reg_lambda', 'subsample', 'colsample_bytree', 'min_child_weight']

BOUNDS_BASE = {
    'n_estimators': (50, 500), 'max_depth': (3, 10), 'learning_rate': (0.01, 0.3),
}
BOUNDS_EXTRA = {
    'reg_alpha': (0.0, 5.0), 'reg_lambda': (0.0, 5.0), 'subsample': (0.5, 1.0),
    'colsample_bytree': (0.5, 1.0), 'min_child_weight': (1, 10),
}

GRID_BASE = {
    'n_estimators': [50, 100, 200, 300, 500],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1, 0.2],
}
GRID_EXTRA = {                         # nilai ringkas agar grid tetap feasible
    'reg_alpha': [0.0, 1.0],
    'reg_lambda': [1.0, 3.0],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 5],
}


def _param_names(cfg):
    return PARAM_ORDER_BASE + (PARAM_EXTRA if cfg['xgb_space'] == 'expanded' else [])


def _bounds_list(cfg):
    names = _param_names(cfg)
    b = {**BOUNDS_BASE, **BOUNDS_EXTRA}
    return [b[n] for n in names]


def _decode_vector(vec, cfg):
    """Vektor kontinu (PSO/GEO) -> dict param XGBoost valid."""
    names = _param_names(cfg)
    p = {}
    for i, name in enumerate(names):
        v = vec[i]
        if name == 'n_estimators':
            p[name] = int(np.clip(round(v / 50) * 50, 50, 500))
        elif name == 'max_depth':
            p[name] = int(np.clip(round(v), 3, 10))
        elif name == 'min_child_weight':
            p[name] = int(np.clip(round(v), 1, 10))
        elif name == 'learning_rate':
            p[name] = float(np.clip(v, 0.01, 0.3))
        elif name in ('subsample', 'colsample_bytree'):
            p[name] = float(np.clip(v, 0.5, 1.0))
        else:                          # reg_alpha, reg_lambda
            p[name] = float(np.clip(v, 0.0, 5.0))
    return p


def _make_xgb(params, cfg):
    kw = dict(objective=cfg.get('xgb_objective', 'reg:squarederror'),
              random_state=SEED, n_jobs=-1, tree_method='hist', device=XGB_DEVICE)
    if cfg.get('xgb_objective') == 'reg:tweedie':
        kw['tweedie_variance_power'] = 1.3
    kw.update(params)
    return xgb.XGBRegressor(**kw)


def _t_fwd(y, cfg):
    return np.log1p(y) if cfg.get('target_transform') == 'log1p' else y


def _t_inv(y, cfg):
    return np.expm1(y) if cfg.get('target_transform') == 'log1p' else y


def _needs_nonneg(cfg):
    return cfg.get('xgb_objective') in ('count:poisson', 'reg:tweedie')


def _fit_predict(params, cfg, scheme, Xtr, ytr, Xval, yval, Xpred):
    """Fit LR+XGB (skema tertentu) di train, prediksi Xpred. Ruang asli (y)."""
    if cfg.get('winsorize') is not None:
        lo, hi = np.percentile(ytr, cfg['winsorize'])
        ytr = np.clip(ytr, lo, hi)

    ytr_t = _t_fwd(ytr, cfg)
    lr = LinearRegression().fit(Xtr, ytr_t)
    lr_pred_pred = lr.predict(Xpred)

    model = _make_xgb(params, cfg)
    fit_kw = {}
    if cfg.get('early_stopping'):
        model.set_params(early_stopping_rounds=20)

    if scheme == 'averaging':
        xgb_target = ytr_t
        if _needs_nonneg(cfg):
            xgb_target = np.clip(xgb_target, 0, None)
        if cfg.get('early_stopping'):
            fit_kw['eval_set'] = [(Xval, _t_fwd(yval, cfg))]
            fit_kw['verbose'] = False
        model.fit(Xtr, xgb_target, **fit_kw)
        pred_t = (lr_pred_pred + model.predict(Xpred)) / 2.0
    else:  # residual
        resid = ytr_t - lr.predict(Xtr)
        if _needs_nonneg(cfg):
            resid = np.clip(resid, 0, None)
        if cfg.get('early_stopping'):
            resid_val = _t_fwd(yval, cfg) - lr.predict(Xval)
            if _needs_nonneg(cfg):
                resid_val = np.clip(resid_val, 0, None)
            fit_kw['eval_set'] = [(Xval, resid_val)]
            fit_kw['verbose'] = False
        model.fit(Xtr, resid, **fit_kw)
        pred_t = lr_pred_pred + model.predict(Xpred)

    return _t_inv(pred_t, cfg)


def _val_rmse(params, cfg, scheme, d):
    try:
        pred = _fit_predict(params, cfg, scheme,
                            d['X_train'], d['y_train'], d['X_val'], d['y_val'], d['X_val'])
        return float(np.sqrt(mean_squared_error(d['y_val'], pred)))
    except Exception:
        return 1e12

In [9]:
def _search_grid(cfg, scheme, d):
    grid = {**GRID_BASE}
    if cfg['xgb_space'] == 'expanded':
        grid = {**grid, **GRID_EXTRA}
    best_p, best_v = None, np.inf
    for params in ParameterGrid(grid):
        v = _val_rmse(params, cfg, scheme, d)
        if v < best_v:
            best_v, best_p = v, params
    return best_p


def _search_optuna(cfg, scheme, d):
    names = _param_names(cfg)

    def objective(trial):
        p = {}
        for n in names:
            lo, hi = {**BOUNDS_BASE, **BOUNDS_EXTRA}[n]
            if n == 'n_estimators':
                p[n] = trial.suggest_int(n, 50, 500, step=50)
            elif n in ('max_depth', 'min_child_weight'):
                p[n] = trial.suggest_int(n, int(lo), int(hi))
            else:
                p[n] = trial.suggest_float(n, lo, hi)
        return _val_rmse(p, cfg, scheme, d)

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=OPT_CFG['optuna_trials'], show_progress_bar=False)
    return study.best_params


def _search_pso(cfg, scheme, d):
    bounds = _bounds_list(cfg)
    lb = np.array([b[0] for b in bounds], dtype=float)
    ub = np.array([b[1] for b in bounds], dtype=float)

    def cost(x):
        return np.array([_val_rmse(_decode_vector(row, cfg), cfg, scheme, d) for row in x])

    opt = GlobalBestPSO(n_particles=OPT_CFG['pso_particles'], dimensions=len(bounds),
                        options={'c1': 0.5, 'c2': 0.3, 'w': 0.9}, bounds=(lb, ub))
    _, best_pos = opt.optimize(cost, iters=OPT_CFG['pso_iters'], verbose=False)
    return _decode_vector(best_pos, cfg)


def _search_geo(cfg, scheme, d):
    bounds = _bounds_list(cfg)

    def obj(vec):
        return _val_rmse(_decode_vector(vec, cfg), cfg, scheme, d)

    geo = GEO(obj, dim=len(bounds), bounds=bounds,
              n_agents=OPT_CFG['geo_agents'], max_iter=OPT_CFG['geo_iters'])
    best_pos, _ = geo.optimize()
    return _decode_vector(best_pos, cfg)


SEARCHERS = {
    'GridSearch': _search_grid,
    'Optuna': _search_optuna,
    'PSO': _search_pso,
    'GEO': _search_geo,
}


def run_experiment(cfg, feature_sets, label, optimizers=None, schemes=None, verbose=True,
                   checkpoint_path=None, resume=True):
    """Jalankan semua optimizer x skema x kategori. Kembalikan DataFrame hasil test.

    Checkpoint & resume:
    - Jika checkpoint_path diberikan, hasil tiap langkah (category, scheme, optimizer)
      langsung disimpan ke CSV checkpoint setelah selesai.
    - Jika notebook dijalankan ulang (mis. setelah koneksi Colab putus), langkah yang
      sudah ada di checkpoint dilewati dan proses lanjut dari langkah berikutnya.
    - Verbose menampilkan timestamp [HH:MM:SS], durasi tiap langkah, dan ETA.
    """
    import time
    import os as _os
    from datetime import datetime

    def _now():
        return datetime.now().strftime('%H:%M:%S')

    def _fmt(sec):
        sec = int(sec)
        h, rem = divmod(sec, 3600)
        m, s = divmod(rem, 60)
        if h:
            return f'{h}h{m:02d}m{s:02d}s'
        if m:
            return f'{m}m{s:02d}s'
        return f'{s}s'

    cfg = {'xgb_space': 'base', 'early_stopping': False, 'target_transform': None,
           'xgb_objective': 'reg:squarederror', 'winsorize': None, **cfg}
    optimizers = optimizers or list(SEARCHERS.keys())
    schemes = schemes or ['averaging', 'residual']

    # --- muat checkpoint (jika ada) ---
    done = {}
    if checkpoint_path is not None and resume and _os.path.exists(checkpoint_path):
        try:
            ckpt_df = pd.read_csv(checkpoint_path)
            for _, r in ckpt_df.iterrows():
                done[(r['category'], r['scheme'], r['optimizer'])] = {
                    'experiment': r.get('experiment', label),
                    'category': r['category'], 'optimizer': r['optimizer'],
                    'scheme': r['scheme'], 'test_MSE': float(r['test_MSE']),
                    'test_RMSE': float(r['test_RMSE']), 'params': r.get('params'),
                }
            print(f'[{_now()}] [RESUME] {len(done)} langkah dimuat dari checkpoint: {checkpoint_path}',
                  flush=True)
        except Exception as exc:
            print(f'[{_now()}] [WARN] gagal baca checkpoint ({exc}); mulai dari awal.', flush=True)
            done = {}

    if checkpoint_path is not None:
        _cp = str(checkpoint_path)
        if _cp.startswith('/content/') and not _cp.startswith('/content/drive/'):
            print(f'[{_now()}] [WARN] checkpoint di {checkpoint_path} EPHEMERAL '
                  f'(hilang saat runtime mati). Simpan di Google Drive agar resume tahan putus koneksi.',
                  flush=True)

    def _save_ckpt(rows):
        if checkpoint_path is not None:
            pd.DataFrame(rows).to_csv(checkpoint_path, index=False)

    n_cat = len(feature_sets)
    total_steps = n_cat * len(schemes) * len(optimizers)
    step = 0
    resumed = 0
    t_start = time.time()
    if verbose:
        print(f'[{_now()}] === [{label}] START | {n_cat} kategori x {len(schemes)} skema x '
              f'{len(optimizers)} optimizer = {total_steps} langkah '
              f'(resume: {len(done)} sudah ada) ===', flush=True)

    rows = []
    for ci, cat in enumerate(feature_sets, 1):
        d = cat['data']
        cat_name = cat['category']
        t_cat = time.time()
        if verbose:
            print(f'[{_now()}] [{label}] ({ci}/{n_cat}) kategori {cat_name} '
                  f'| n_features={cat.get("n_features", "?")} '
                  f'| train={len(d["y_train"])} val={len(d["y_val"])} test={len(d["y_test"])}',
                  flush=True)
        for scheme in schemes:
            for opt_name in optimizers:
                step += 1
                key = (cat_name, scheme, opt_name)
                if key in done:
                    row = done[key]
                    row['experiment'] = label
                    rows.append(row)
                    resumed += 1
                    if verbose:
                        print(f'[{_now()}]   ~~ [{label}] {cat_name} | {scheme:>9} | {opt_name:<10} '
                              f'({step}/{total_steps}) SKIP (checkpoint) '
                              f'| test_RMSE={row["test_RMSE"]:.3f}', flush=True)
                    continue
                t0 = time.time()
                if verbose:
                    print(f'[{_now()}]   -> [{label}] {cat_name} | {scheme:>9} | {opt_name:<10} '
                          f'({step}/{total_steps}) start ...', flush=True)
                best_p = SEARCHERS[opt_name](cfg, scheme, d)
                pred_test = _fit_predict(best_p, cfg, scheme,
                                         d['X_train'], d['y_train'],
                                         d['X_val'], d['y_val'], d['X_test'])
                mse = float(mean_squared_error(d['y_test'], pred_test))
                row = {
                    'experiment': label, 'category': cat_name,
                    'optimizer': opt_name, 'scheme': scheme,
                    'test_MSE': mse, 'test_RMSE': float(np.sqrt(mse)),
                    'params': best_p,
                }
                rows.append(row)
                _save_ckpt(rows)  # simpan langsung tiap langkah
                if verbose:
                    elapsed = time.time() - t_start
                    new_done = step - resumed
                    eta = (elapsed / new_done) * (total_steps - step) if new_done else 0
                    print(f'[{_now()}]   <- [{label}] {cat_name} | {scheme:>9} | {opt_name:<10} '
                          f'({step}/{total_steps}) DONE in {_fmt(time.time() - t0)} '
                          f'| test_RMSE={np.sqrt(mse):.3f} test_MSE={mse:.3f} '
                          f'| elapsed {_fmt(elapsed)} | ETA {_fmt(eta)}', flush=True)
        if verbose:
            elapsed = time.time() - t_start
            print(f'[{_now()}] [{label}] kategori {cat_name} SELESAI dalam '
                  f'{_fmt(time.time() - t_cat)} | total berjalan {_fmt(elapsed)}', flush=True)
    if verbose:
        print(f'[{_now()}] === [{label}] SELESAI TOTAL dalam {_fmt(time.time() - t_start)} '
              f'({len(rows)} baris hasil, {resumed} dari checkpoint) ===', flush=True)
    return pd.DataFrame(rows)

def show_result(res_df, baseline_df=None):
    """Tampilkan tabel ringkas + delta test_MSE vs baseline (best per kategori)."""
    view = res_df[['category', 'optimizer', 'scheme', 'test_MSE', 'test_RMSE']].copy()
    if baseline_df is not None:
        base_best = baseline_df.groupby('category')['test_MSE'].min()
        best = res_df.loc[res_df.groupby('category')['test_MSE'].idxmin()].copy()
        best['baseline_MSE'] = best['category'].map(base_best)
        best['delta_MSE'] = best['test_MSE'] - best['baseline_MSE']
        print('== Best per kategori vs baseline ==')
        display(best[['category', 'optimizer', 'scheme', 'test_MSE', 'baseline_MSE', 'delta_MSE']]
                .reset_index(drop=True))
    return view.sort_values(['category', 'test_MSE']).reset_index(drop=True)

# Final Combination Setup

Dua feature set dibuat:
- `feat_base`: fitur ACF dasar untuk kategori yang tetap memakai baseline.
- `feat_seasonal`: fitur ACF + seasonal untuk kategori yang memakai enhancement.

In [10]:
feat_base = make_splits(data, seasonal=False)
feat_seasonal = make_splits(data, seasonal=True)

FEATURE_SETS = {
    'base': {item['category']: item for item in feat_base},
    'seasonal': {item['category']: item for item in feat_seasonal},
}

print('Base features:', {c['category']: c['n_features'] for c in feat_base})
print('Seasonal features:', {c['category']: c['n_features'] for c in feat_seasonal})

Base features: {'M01AB': 25, 'M01AE': 4, 'N02BA': 27, 'N02BE': 23, 'N05B': 16, 'N05C': 3, 'R03': 16, 'R06': 24}
Seasonal features: {'M01AB': 38, 'M01AE': 17, 'N02BA': 40, 'N02BE': 36, 'N05B': 29, 'N05C': 16, 'R03': 29, 'R06': 37}


## Baseline Reference

Jika `weekly_baseline_results.csv` tersedia di `OUTPUT_DIR`, notebook akan memuatnya
untuk menghitung delta MSE/RMSE vs baseline.

In [11]:
BASELINE_RESULT_PATH = OUTPUT_DIR / 'weekly_baseline_results.csv'
if BASELINE_RESULT_PATH.exists():
    baseline_ref = pd.read_csv(BASELINE_RESULT_PATH)
    print(f'Baseline loaded: {BASELINE_RESULT_PATH}')
else:
    baseline_ref = None
    print(f'Baseline CSV belum ada: {BASELINE_RESULT_PATH}. Delta vs baseline tidak ditampilkan.')

Baseline loaded: /content/drive/MyDrive/pharma_weekly_enhancement_outputs/weekly_baseline_results.csv


# Final Config Per Kategori

Konfigurasi dipilih dari hasil summary penuh:

| Kategori | Config | Optimizer | Scheme | Feature set |
|---|---|---|---|---|
| M01AB | pseudo-huber + expanded | Optuna | averaging | seasonal |
| M01AE | log1p + expanded | GEO | residual | seasonal |
| N02BA | poisson + expanded | GEO | averaging | seasonal |
| N02BE | log1p + expanded | GEO | averaging | seasonal |
| N05B | baseline | GEO | averaging | base |
| N05C | pseudo-huber + expanded | GEO | averaging | seasonal |
| R03 | winsorize + expanded | PSO | residual | seasonal |
| R06 | pseudo-huber + expanded | PSO | averaging | seasonal |

In [12]:
FINAL_CONFIGS = [
    {
        'category': 'M01AB', 'experiment': 'F_final_combo', 'feature_set': 'seasonal',
        'optimizer': 'Optuna', 'scheme': 'averaging',
        'cfg': {'xgb_space': 'expanded', 'xgb_objective': 'reg:pseudohubererror'},
        'reason': 'Best overall: E2_pseudohuber + Optuna + averaging',
    },
    {
        'category': 'M01AE', 'experiment': 'F_final_combo', 'feature_set': 'seasonal',
        'optimizer': 'GEO', 'scheme': 'residual',
        'cfg': {'xgb_space': 'expanded', 'target_transform': 'log1p'},
        'reason': 'Best overall: D1_log1p + GEO + residual',
    },
    {
        'category': 'N02BA', 'experiment': 'F_final_combo', 'feature_set': 'seasonal',
        'optimizer': 'GEO', 'scheme': 'averaging',
        'cfg': {'xgb_space': 'expanded', 'xgb_objective': 'count:poisson'},
        'reason': 'Best overall: D2_poisson + GEO + averaging',
    },
    {
        'category': 'N02BE', 'experiment': 'F_final_combo', 'feature_set': 'seasonal',
        'optimizer': 'GEO', 'scheme': 'averaging',
        'cfg': {'xgb_space': 'expanded', 'target_transform': 'log1p'},
        'reason': 'Best overall: D1_log1p + GEO + averaging',
    },
    {
        'category': 'N05B', 'experiment': 'F_final_combo', 'feature_set': 'base',
        'optimizer': 'GEO', 'scheme': 'averaging',
        'cfg': {'xgb_space': 'base'},
        'reason': 'Baseline tetap terbaik; enhancement memperburuk',
    },
    {
        'category': 'N05C', 'experiment': 'F_final_combo', 'feature_set': 'seasonal',
        'optimizer': 'GEO', 'scheme': 'averaging',
        'cfg': {'xgb_space': 'expanded', 'xgb_objective': 'reg:pseudohubererror'},
        'reason': 'Best overall: E2_pseudohuber + GEO + averaging',
    },
    {
        'category': 'R03', 'experiment': 'F_final_combo', 'feature_set': 'seasonal',
        'optimizer': 'PSO', 'scheme': 'residual',
        'cfg': {'xgb_space': 'expanded', 'winsorize': (1, 99)},
        'reason': 'Best overall: E1_winsorize + PSO + residual',
    },
    {
        'category': 'R06', 'experiment': 'F_final_combo', 'feature_set': 'seasonal',
        'optimizer': 'PSO', 'scheme': 'averaging',
        'cfg': {'xgb_space': 'expanded', 'xgb_objective': 'reg:pseudohubererror'},
        'reason': 'Best overall: E2_pseudohuber + PSO + averaging',
    },
]

pd.DataFrame([{k: v for k, v in row.items() if k != 'cfg'} | {'cfg': row['cfg']} for row in FINAL_CONFIGS])

,category,experiment,feature_set,optimizer,scheme,reason,cfg
0,M01AB,F_final_combo,seasonal,Optuna,averaging,Best overall: E2_pseudohuber + Optuna + averaging,"{'xgb_space': 'expanded', 'xgb_objective': 're..."
1,M01AE,F_final_combo,seasonal,GEO,residual,Best overall: D1_log1p + GEO + residual,"{'xgb_space': 'expanded', 'target_transform': ..."
2,N02BA,F_final_combo,seasonal,GEO,averaging,Best overall: D2_poisson + GEO + averaging,"{'xgb_space': 'expanded', 'xgb_objective': 'co..."
3,N02BE,F_final_combo,seasonal,GEO,averaging,Best overall: D1_log1p + GEO + averaging,"{'xgb_space': 'expanded', 'target_transform': ..."
4,N05B,F_final_combo,base,GEO,averaging,Baseline tetap terbaik; enhancement memperburuk,{'xgb_space': 'base'}
5,N05C,F_final_combo,seasonal,GEO,averaging,Best overall: E2_pseudohuber + GEO + averaging,"{'xgb_space': 'expanded', 'xgb_objective': 're..."
6,R03,F_final_combo,seasonal,PSO,residual,Best overall: E1_winsorize + PSO + residual,"{'xgb_space': 'expanded', 'winsorize': (1, 99)}"
7,R06,F_final_combo,seasonal,PSO,averaging,Best overall: E2_pseudohuber + PSO + averaging,"{'xgb_space': 'expanded', 'xgb_objective': 're..."


# Runner: Final Combination With Checkpoint

Checkpoint disimpan per kategori di `weekly_F_final_combo_checkpoint.csv`.
Jika Colab/network timeout, run ulang notebook ini; kategori yang sudah selesai akan
di-`SKIP (checkpoint)` dan proses lanjut dari kategori berikutnya.

In [13]:
def run_final_combination(final_configs, feature_sets, checkpoint_path=None, resume=True, verbose=True):
    import time
    import os as _os
    from datetime import datetime

    def _now():
        return datetime.now().strftime('%H:%M:%S')

    def _fmt(sec):
        sec = int(sec)
        h, rem = divmod(sec, 3600)
        m, s = divmod(rem, 60)
        if h:
            return f'{h}h{m:02d}m{s:02d}s'
        if m:
            return f'{m}m{s:02d}s'
        return f'{s}s'

    done = {}
    if checkpoint_path is not None and resume and _os.path.exists(checkpoint_path):
        ckpt_df = pd.read_csv(checkpoint_path)
        for _, r in ckpt_df.iterrows():
            done[str(r['category'])] = r.to_dict()
        print(f'[{_now()}] [RESUME] {len(done)} kategori dimuat dari checkpoint: {checkpoint_path}', flush=True)

    if checkpoint_path is not None:
        cp = str(checkpoint_path)
        if cp.startswith('/content/') and not cp.startswith('/content/drive/'):
            print(f'[{_now()}] [WARN] checkpoint di {checkpoint_path} EPHEMERAL; gunakan Drive agar resume tahan runtime reset.', flush=True)

    def _save(rows):
        if checkpoint_path is not None:
            pd.DataFrame(rows).to_csv(checkpoint_path, index=False)

    rows = []
    total = len(final_configs)
    t_start = time.time()
    if verbose:
        print(f'[{_now()}] === [F_final_combo] START | {total} kategori | resume: {len(done)} sudah ada ===', flush=True)

    for i, item in enumerate(final_configs, 1):
        cat = item['category']
        if cat in done:
            row = done[cat]
            rows.append(row)
            if verbose:
                print(f'[{_now()}] ~~ {cat} ({i}/{total}) SKIP (checkpoint) | '
                      f'test_RMSE={float(row["test_RMSE"]):.3f} test_MSE={float(row["test_MSE"]):.3f}', flush=True)
            continue

        feature_key = item['feature_set']
        d = feature_sets[feature_key][cat]['data']
        cfg = item['cfg']
        opt_name = item['optimizer']
        scheme = item['scheme']
        t0 = time.time()
        if verbose:
            print(f'[{_now()}] -> {cat} ({i}/{total}) | feature={feature_key} | '
                  f'optimizer={opt_name} | scheme={scheme} | cfg={cfg} start ...', flush=True)

        best_p = SEARCHERS[opt_name](cfg, scheme, d)
        pred_test = _fit_predict(best_p, {'xgb_space': 'base', 'early_stopping': False,
                                          'target_transform': None, 'xgb_objective': 'reg:squarederror',
                                          'winsorize': None, **cfg},
                                 scheme, d['X_train'], d['y_train'], d['X_val'], d['y_val'], d['X_test'])
        mse = float(mean_squared_error(d['y_test'], pred_test))
        row = {
            'experiment': 'F_final_combo',
            'category': cat,
            'feature_set': feature_key,
            'optimizer': opt_name,
            'scheme': scheme,
            'test_MSE': mse,
            'test_RMSE': float(np.sqrt(mse)),
            'params': best_p,
            'cfg': cfg,
            'reason': item['reason'],
        }
        rows.append(row)
        _save(rows)

        if verbose:
            elapsed = time.time() - t_start
            eta = (elapsed / i) * (total - i)
            print(f'[{_now()}] <- {cat} ({i}/{total}) DONE in {_fmt(time.time() - t0)} | '
                  f'test_RMSE={np.sqrt(mse):.3f} test_MSE={mse:.3f} | '
                  f'elapsed {_fmt(elapsed)} | ETA {_fmt(eta)}', flush=True)

    if verbose:
        print(f'[{_now()}] === [F_final_combo] SELESAI dalam {_fmt(time.time() - t_start)} '
              f'({len(rows)} baris hasil) ===', flush=True)
    return pd.DataFrame(rows)

# Run Final Combination

In [14]:
CKPT_PATH = OUTPUT_DIR / 'weekly_F_final_combo_checkpoint.csv'
res_F = run_final_combination(FINAL_CONFIGS, FEATURE_SETS, checkpoint_path=CKPT_PATH)
res_F

[00:59:43] === [F_final_combo] START | 8 kategori | resume: 0 sudah ada ===
[00:59:43] -> M01AB (1/8) | feature=seasonal | optimizer=Optuna | scheme=averaging | cfg={'xgb_space': 'expanded', 'xgb_objective': 'reg:pseudohubererror'} start ...
[00:59:53] <- M01AB (1/8) DONE in 9s | test_RMSE=7.760 test_MSE=60.214 | elapsed 9s | ETA 1m09s
[00:59:53] -> M01AE (2/8) | feature=seasonal | optimizer=GEO | scheme=residual | cfg={'xgb_space': 'expanded', 'target_transform': 'log1p'} start ...
[01:00:19] <- M01AE (2/8) DONE in 25s | test_RMSE=8.222 test_MSE=67.602 | elapsed 35s | ETA 1m46s
[01:00:19] -> N02BA (3/8) | feature=seasonal | optimizer=GEO | scheme=averaging | cfg={'xgb_space': 'expanded', 'xgb_objective': 'count:poisson'} start ...
[01:05:22] <- N02BA (3/8) DONE in 5m02s | test_RMSE=5.984 test_MSE=35.809 | elapsed 5m38s | ETA 9m23s
[01:05:22] -> N02BE (4/8) | feature=seasonal | optimizer=GEO | scheme=averaging | cfg={'xgb_space': 'expanded', 'target_transform': 'log1p'} start ...
[01:0

,experiment,category,feature_set,optimizer,scheme,test_MSE,test_RMSE,params,cfg,reason
0,F_final_combo,M01AB,seasonal,Optuna,averaging,60.213724,7.759750,"{'n_estimators': 500, 'max_depth': 3, 'learnin...","{'xgb_space': 'expanded', 'xgb_objective': 're...",Best overall: E2_pseudohuber + Optuna + averaging
1,F_final_combo,M01AE,seasonal,GEO,residual,67.601516,8.222014,"{'n_estimators': 150, 'max_depth': 7, 'learnin...","{'xgb_space': 'expanded', 'target_transform': ...",Best overall: D1_log1p + GEO + residual
2,F_final_combo,N02BA,seasonal,GEO,averaging,35.808738,5.984040,"{'n_estimators': 500, 'max_depth': 10, 'learni...","{'xgb_space': 'expanded', 'xgb_objective': 'co...",Best overall: D2_poisson + GEO + averaging
3,F_final_combo,N02BE,seasonal,GEO,averaging,2358.274989,48.562074,"{'n_estimators': 100, 'max_depth': 9, 'learnin...","{'xgb_space': 'expanded', 'target_transform': ...",Best overall: D1_log1p + GEO + averaging
4,F_final_combo,N05B,base,GEO,averaging,143.224445,11.967642,"{'n_estimators': 50, 'max_depth': 4, 'learning...",{'xgb_space': 'base'},Baseline tetap terbaik; enhancement memperburuk
5,F_final_combo,N05C,seasonal,GEO,averaging,7.340645,2.709363,"{'n_estimators': 50, 'max_depth': 4, 'learning...","{'xgb_space': 'expanded', 'xgb_objective': 're...",Best overall: E2_pseudohuber + GEO + averaging
6,F_final_combo,R03,seasonal,PSO,residual,836.099261,28.915381,"{'n_estimators': 250, 'max_depth': 7, 'learnin...","{'xgb_space': 'expanded', 'winsorize': (1, 99)}",Best overall: E1_winsorize + PSO + residual
7,F_final_combo,R06,seasonal,PSO,averaging,75.641422,8.697208,"{'n_estimators': 300, 'max_depth': 7, 'learnin...","{'xgb_space': 'expanded', 'xgb_objective': 're...",Best overall: E2_pseudohuber + PSO + averaging


# Compare vs Baseline

In [15]:
if baseline_ref is not None:
    base_best = baseline_ref.groupby('category')['test_MSE'].min()
    compare_F = res_F.copy()
    compare_F['baseline_MSE'] = compare_F['category'].map(base_best)
    compare_F['delta_MSE'] = compare_F['test_MSE'] - compare_F['baseline_MSE']
    compare_F['improvement_pct'] = ((compare_F['baseline_MSE'] - compare_F['test_MSE']) / compare_F['baseline_MSE'] * 100).round(2)
    display(compare_F[['category', 'feature_set', 'optimizer', 'scheme', 'test_MSE', 'test_RMSE',
                       'baseline_MSE', 'delta_MSE', 'improvement_pct', 'reason']]
            .sort_values('category').reset_index(drop=True))
else:
    compare_F = res_F.copy()
    display(compare_F[['category', 'feature_set', 'optimizer', 'scheme', 'test_MSE', 'test_RMSE', 'reason']]
            .sort_values('category').reset_index(drop=True))

,category,feature_set,optimizer,scheme,test_MSE,test_RMSE,baseline_MSE,delta_MSE,improvement_pct,reason
0,M01AB,seasonal,Optuna,averaging,60.213724,7.759750,67.020749,-6.807025,10.16,Best overall: E2_pseudohuber + Optuna + averaging
1,M01AE,seasonal,GEO,residual,67.601516,8.222014,69.474636,-1.873121,2.70,Best overall: D1_log1p + GEO + residual
2,N02BA,seasonal,GEO,averaging,35.808738,5.984040,38.715477,-2.906738,7.51,Best overall: D2_poisson + GEO + averaging
3,N02BE,seasonal,GEO,averaging,2358.274989,48.562074,2886.454217,-528.179228,18.30,Best overall: D1_log1p + GEO + averaging
4,N05B,base,GEO,averaging,143.224445,11.967642,143.225024,-0.000579,0.00,Baseline tetap terbaik; enhancement memperburuk
5,N05C,seasonal,GEO,averaging,7.340645,2.709363,7.379361,-0.038715,0.52,Best overall: E2_pseudohuber + GEO + averaging
6,R03,seasonal,PSO,residual,836.099261,28.915381,721.610398,114.488863,-15.87,Best overall: E1_winsorize + PSO + residual
7,R06,seasonal,PSO,averaging,75.641422,8.697208,71.428755,4.212667,-5.90,Best overall: E2_pseudohuber + PSO + averaging


# Save Final Result

Hasil final disimpan sebagai `weekly_F_final_combo_results.csv`. Setelah berhasil
disimpan, checkpoint dihapus agar run berikutnya bersih. CSV hasil final tetap aman.

In [16]:
RESULT_PATH = OUTPUT_DIR / 'weekly_F_final_combo_results.csv'
res_F.to_csv(RESULT_PATH, index=False)
print('Tersimpan:', RESULT_PATH)

if 'compare_F' in globals():
    COMPARE_PATH = OUTPUT_DIR / 'weekly_F_final_combo_compare.csv'
    compare_F.to_csv(COMPARE_PATH, index=False)
    print('Comparison tersimpan:', COMPARE_PATH)

try:
    if 'CKPT_PATH' in globals() and CKPT_PATH.exists():
        CKPT_PATH.unlink()
        print('Checkpoint dihapus:', CKPT_PATH)
except Exception as exc:
    print('[WARN] gagal hapus checkpoint:', exc)

Tersimpan: /content/drive/MyDrive/pharma_weekly_enhancement_outputs/weekly_F_final_combo_results.csv
Comparison tersimpan: /content/drive/MyDrive/pharma_weekly_enhancement_outputs/weekly_F_final_combo_compare.csv
Checkpoint dihapus: /content/drive/MyDrive/pharma_weekly_enhancement_outputs/weekly_F_final_combo_checkpoint.csv
